# Experiment 3 — Quantum Teleportation and QKD Simulation
**Part A:** Teleport a qubit from Alice to Bob using entanglement.
**Part B:** BB84 QKD protocol — generate a shared secret key.

In [ ]:
!pip install qiskit qiskit-aer matplotlib numpy -q

## Part A — Quantum Teleportation

## Imports

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import random_statevector, Statevector
from qiskit.visualization import plot_bloch_multivector
import matplotlib.pyplot as plt

### Step 1 — Create Bell Pair (Entanglement between Alice and Bob)

In [ ]:
def make_bell_pair(circuit, qubit_a, qubit_b):
    circuit.h(qubit_a)
    circuit.cx(qubit_a, qubit_b)

### Step 2 — Alice's Gates (CNOT + H on her qubits)

In [ ]:
def alice_operations(circuit, psi, qubit_a):
    circuit.cx(psi, qubit_a)
    circuit.h(psi)

### Step 3 — Alice Measures and Sends 2 Classical Bits to Bob

In [ ]:
def alice_measure_and_send(circuit, psi, qubit_a, bit_z, bit_x):
    circuit.barrier()
    circuit.measure(psi, bit_z)
    circuit.measure(qubit_a, bit_x)

### Step 4 — Bob Applies Correction Gates Based on Classical Bits

In [ ]:
def bob_decode(circuit, qubit_b, bit_z, bit_x):
    circuit.x(qubit_b).c_if(bit_x, 1)
    circuit.z(qubit_b).c_if(bit_z, 1)

### Step 5 — Build and Run Full Teleportation Circuit

In [ ]:
psi_to_send = random_statevector(2)
print(f"State Alice wants to teleport:\n  {psi_to_send}")

qregs  = QuantumRegister(3, 'q')
bit_z  = ClassicalRegister(1, 'crz')
bit_x  = ClassicalRegister(1, 'crx')
tele   = QuantumCircuit(qregs, bit_z, bit_x)

tele.initialize(psi_to_send.data, 0)
tele.barrier()

make_bell_pair(tele, 1, 2)
tele.barrier()

alice_operations(tele, 0, 1)
alice_measure_and_send(tele, 0, 1, bit_z[0], bit_x[0])
tele.barrier()

bob_decode(tele, 2, bit_z[0], bit_x[0])

print("\nTeleportation Circuit:")
print(tele.draw('text'))

### Step 6 — Verify Bob's Qubit Matches Alice's Original State

In [ ]:
verify = QuantumCircuit(qregs, bit_z, bit_x)
verify.initialize(psi_to_send.data, 0)
verify.barrier()
make_bell_pair(verify, 1, 2)
verify.barrier()
alice_operations(verify, 0, 1)
alice_measure_and_send(verify, 0, 1, bit_z[0], bit_x[0])
verify.barrier()
bob_decode(verify, 2, bit_z[0], bit_x[0])
verify.save_statevector()

machine  = AerSimulator()
outcome  = machine.run(transpile(verify, machine)).result()
sv       = outcome.get_statevector()

fig = plot_bloch_multivector(sv)
fig.suptitle("q0 and q1 collapsed. q2 (Bob) = original psi", fontsize=10)
plt.show()
print("q2 arrow on Bloch sphere should match the original psi direction.")

---
## Part B — QKD BB84 Protocol

## Imports

In [ ]:
import numpy as np

### Configuration

In [ ]:
number_of_qubits  = 50
check_fraction    = 0.5

### Step 1 — Alice Generates Bits and Encodes Photons

In [ ]:
def alice_prepare(length):
    secret_bits   = np.random.randint(2, size=length)
    chosen_bases  = np.random.randint(2, size=length)
    return secret_bits, chosen_bases

def encode_photons(bits, bases):
    photons = []
    for bit, base in zip(bits, bases):
        if base == 0:
            photons.append('H' if bit == 0 else 'V')
        else:
            photons.append('D' if bit == 0 else 'A')
    return np.array(photons)

### Step 2 — Bob Measures in Random Bases

In [ ]:
def bob_measure(photons, bob_bases):
    bob_bits = []
    for photon, base in zip(photons, bob_bases):
        is_straight = (photon in ('H', 'V'))
        alice_base  = 0 if is_straight else 1
        if base == alice_base:
            bob_bits.append(0 if photon in ('H', 'D') else 1)
        else:
            bob_bits.append(np.random.randint(2))
    return np.array(bob_bits)

### Step 3 — Sifting (Keep Only Matching Basis Bits)

In [ ]:
def sift(alice_bits, alice_bases, bob_bits, bob_bases):
    matched         = (alice_bases == bob_bases)
    alice_sifted    = alice_bits[matched]
    bob_sifted      = bob_bits[matched]
    return alice_sifted, bob_sifted

### Step 4 — Error Check (QBER)

In [ ]:
def check_errors(alice_key, bob_key, fraction):
    n          = len(alice_key)
    check_size = int(n * fraction)
    if check_size == 0:
        return 0, alice_key, bob_key

    check_idx   = np.random.choice(n, size=check_size, replace=False)
    keep_idx    = np.setdiff1d(np.arange(n), check_idx)

    errors      = np.sum(alice_key[check_idx] != bob_key[check_idx])
    qber        = errors / check_size

    return qber, alice_key[keep_idx], bob_key[keep_idx]

### Step 5 — Run Full BB84 Protocol (No Eve)

In [ ]:
def run_bb84():
    print("--- BB84 QKD Protocol Simulation ---")
    print(f"Qubits sent: {number_of_qubits}\n")

    alice_bits, alice_bases = alice_prepare(number_of_qubits)
    photons                 = encode_photons(alice_bits, alice_bases)

    print("[Step 1: Alice's Preparation]")
    print(f"Alice's Bits:  {alice_bits}")
    print(f"Alice's Bases: {alice_bases}  (0=+, 1=x)")
    print(f"Photons Sent:  {photons}")

    bob_bases = np.random.randint(2, size=number_of_qubits)
    bob_bits  = bob_measure(photons, bob_bases)

    print("\n[Step 2: Bob's Measurement]")
    print(f"Bob's Bases: {bob_bases}")
    print(f"Bob's Bits:  {bob_bits}")

    alice_sifted, bob_sifted = sift(alice_bits, alice_bases, bob_bits, bob_bases)

    print("\n[Step 3: Sifting]")
    print(f"Sifted key length: {len(alice_sifted)} bits")
    print(f"Alice sifted: {alice_sifted}")
    print(f"Bob sifted:   {bob_sifted}")

    qber, alice_final, bob_final = check_errors(alice_sifted, bob_sifted, check_fraction)

    print("\n[Step 4: Error Check]")
    print(f"QBER: {qber*100:.2f}%")
    print(f"Final key length: {len(alice_final)} bits")

    if qber == 0:
        print("\n[SUCCESS] No eavesdropping detected.")
        print(f"Final Secret Key: {alice_final}")
    else:
        print(f"\n[FAILURE] QBER = {qber*100:.2f}%. Key discarded.")

run_bb84()

### Step 6 — Simulate Eve Intercepting

In [ ]:
def run_bb84_with_eve():
    print("--- BB84 With Eve ---")
    alice_bits, alice_bases = alice_prepare(number_of_qubits)
    photons                 = encode_photons(alice_bits, alice_bases)

    eve_bases  = np.random.randint(2, size=number_of_qubits)
    eve_bits   = bob_measure(photons, eve_bases)
    eve_resent = encode_photons(eve_bits, eve_bases)

    bob_bases = np.random.randint(2, size=number_of_qubits)
    bob_bits  = bob_measure(eve_resent, bob_bases)

    alice_sifted, bob_sifted = sift(alice_bits, alice_bases, bob_bits, bob_bases)
    qber, _, _               = check_errors(alice_sifted, bob_sifted, check_fraction)

    print(f"QBER with Eve: {qber*100:.2f}%  (expected ~25%)")
    print("Eve DETECTED — Key Discarded." if qber > 0.11 else "Eve not caught this run.")

run_bb84_with_eve()